# CrossRef Author Works Query

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MattArtzAnthro/wikidata-tools/blob/main/notebooks/CrossRef_Author_Works_Query.ipynb)

Created by [Matt Artz](https://www.mattartz.me/) | [GitHub](https://github.com/MattArtzAnthro) | [ORCID](https://orcid.org/0000-0002-3822-1429)

---

## What This Notebook Does

This notebook queries the CrossRef API to retrieve comprehensive metadata for all works by a specific author. It supports two query methods: (1) **ORCID-based search** to find all works linked to an author's ORCID identifier, and (2) **DOI list lookup** to fetch metadata for a specific set of known DOIs.

The output includes all available CrossRef fields in a structured format suitable for Wikidata import or bibliometric analysis: titles, abstracts, author details with affiliations and ORCIDs, publication dates, journal information, volume/issue/pages, references, funders, licenses, and more.

## Key Features

- **Dual Query Modes**: Search by ORCID or batch lookup by DOI list
- **Complete Field Extraction**: Captures all CrossRef metadata fields
- **Author Detail Parsing**: Extracts names, ORCIDs, affiliations, and sequence positions
- **Reference Extraction**: Optionally includes cited references with DOIs
- **Structured Export**: CSV with flattened fields plus JSON with full nested structure

## Workflow

1. **Setup**: Install dependencies and configure API settings
2. **Input**: Enter ORCID or upload DOI list
3. **Query**: Fetch works from CrossRef API with rate limiting
4. **Parse**: Extract and structure all metadata fields
5. **Export**: Download CSV and JSON outputs

## Citation

> Artz, M. (2026). Wikidata Tools. GitHub. https://github.com/MattArtzAnthro/wikidata-tools

*A citable DOI will be available via Zenodo.*

## License

[CC BY-NC 4.0](https://creativecommons.org/licenses/by-nc/4.0/)

## Setup

In [ ]:
# Install required packages
!pip install requests pandas ipywidgets -q

import requests
import pandas as pd
import json
import re
import time
import os
from datetime import datetime
from typing import Dict, List, Optional, Tuple, Any
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets
from io import BytesIO, StringIO

# Google Colab file handling
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("✅ Setup complete.")
print(f"   Running in Colab: {IN_COLAB}")

## Configuration

In [ ]:
# CrossRef API Configuration
class Config:
    # API settings
    CROSSREF_API_BASE = "https://api.crossref.org"
    USER_AGENT = "CrossRefAuthorWorksQuery/1.0 (https://github.com/mattartz; mailto:matt@mattartz.me)"
    
    # Rate limiting (CrossRef requests polite pool access)
    REQUEST_DELAY = 1.0  # seconds between requests
    TIMEOUT = 60  # seconds
    
    # Pagination
    ROWS_PER_PAGE = 100  # max 1000, but 100 is more reliable
    MAX_PAGES = 50  # safety limit
    
    # Output
    OUTPUT_PATH = "/content/outputs/" if IN_COLAB else "outputs/"
    INCLUDE_REFERENCES = True  # include cited references (increases data size)
    INCLUDE_ABSTRACTS = True

config = Config()

# Create output directory
os.makedirs(config.OUTPUT_PATH, exist_ok=True)

print("⚙️ Configuration")
print("=" * 50)
print(f"CrossRef API: {config.CROSSREF_API_BASE}")
print(f"Request delay: {config.REQUEST_DELAY}s")
print(f"Rows per page: {config.ROWS_PER_PAGE}")
print(f"Include references: {config.INCLUDE_REFERENCES}")
print(f"Include abstracts: {config.INCLUDE_ABSTRACTS}")
print(f"Output path: {config.OUTPUT_PATH}")

## Helper Functions

In [ ]:
def clean_doi(doi_input: str) -> Optional[str]:
    """Normalize DOI to bare format (e.g., 10.1234/example)."""
    if not doi_input or not isinstance(doi_input, str):
        return None
    doi_input = str(doi_input).strip()
    
    # Extract DOI from URLs
    for pattern in [r'https?://(?:dx\.)?doi\.org/(.+)', r'doi:(.+)']:
        match = re.search(pattern, doi_input, re.IGNORECASE)
        if match:
            doi_input = match.group(1)
            break
    
    # Validate DOI format
    if re.match(r'^10\.\d+/.+', doi_input):
        return doi_input.strip()
    return None


def clean_orcid(orcid_input: str) -> Optional[str]:
    """Normalize ORCID to bare format (e.g., 0000-0002-1825-0097)."""
    if not orcid_input or not isinstance(orcid_input, str):
        return None
    orcid_input = str(orcid_input).strip()
    
    # Extract from URL format
    match = re.search(r'(\d{4}-\d{4}-\d{4}-\d{3}[\dX])', orcid_input, re.IGNORECASE)
    if match:
        return match.group(1).upper()
    return None


def parse_date_parts(date_parts: List) -> Tuple[Optional[str], Optional[int], Optional[int], Optional[int]]:
    """
    Parse CrossRef date-parts array into components.
    Returns: (iso_date, year, month, day)
    """
    if not date_parts or not isinstance(date_parts, list):
        return None, None, None, None
    
    parts = date_parts[0] if date_parts else []
    year = parts[0] if len(parts) > 0 else None
    month = parts[1] if len(parts) > 1 else None
    day = parts[2] if len(parts) > 2 else None
    
    # Build ISO date string
    if year:
        if month and day:
            iso_date = f"{year:04d}-{month:02d}-{day:02d}"
        elif month:
            iso_date = f"{year:04d}-{month:02d}"
        else:
            iso_date = f"{year:04d}"
    else:
        iso_date = None
    
    return iso_date, year, month, day


def parse_authors(author_list: List[Dict]) -> List[Dict]:
    """
    Parse CrossRef author array into structured list.
    """
    if not author_list:
        return []
    
    authors = []
    for idx, author in enumerate(author_list):
        author_data = {
            'sequence_number': idx + 1,
            'sequence': author.get('sequence', ''),  # 'first' or 'additional'
            'given': author.get('given', ''),
            'family': author.get('family', ''),
            'name': author.get('name', ''),  # for organizations
            'orcid': clean_orcid(author.get('ORCID', '')),
            'authenticated_orcid': author.get('authenticated-orcid', False),
            'suffix': author.get('suffix', ''),
            'affiliation': '; '.join([aff.get('name', '') for aff in author.get('affiliation', [])]),
            'affiliation_ids': '; '.join([
                f"{aff_id.get('id-type', '')}:{aff_id.get('id', '')}" 
                for aff in author.get('affiliation', []) 
                for aff_id in aff.get('id', [])
            ] if author.get('affiliation') else [])
        }
        
        # Build display name
        if author_data['given'] and author_data['family']:
            author_data['display_name'] = f"{author_data['given']} {author_data['family']}"
        elif author_data['family']:
            author_data['display_name'] = author_data['family']
        elif author_data['name']:
            author_data['display_name'] = author_data['name']
        else:
            author_data['display_name'] = ''
            
        authors.append(author_data)
    
    return authors


def parse_references(reference_list: List[Dict]) -> List[Dict]:
    """
    Parse CrossRef reference array.
    """
    if not reference_list:
        return []
    
    refs = []
    for ref in reference_list:
        ref_data = {
            'key': ref.get('key', ''),
            'doi': clean_doi(ref.get('DOI', '')),
            'unstructured': ref.get('unstructured', ''),
            'article_title': ref.get('article-title', ''),
            'volume': ref.get('volume', ''),
            'first_page': ref.get('first-page', ''),
            'year': ref.get('year', ''),
            'journal_title': ref.get('journal-title', ''),
            'author': ref.get('author', ''),
            'issn': ref.get('ISSN', ''),
            'isbn': ref.get('ISBN', '')
        }
        refs.append(ref_data)
    
    return refs


def parse_funders(funder_list: List[Dict]) -> List[Dict]:
    """
    Parse CrossRef funder array.
    """
    if not funder_list:
        return []
    
    funders = []
    for funder in funder_list:
        funder_data = {
            'name': funder.get('name', ''),
            'doi': funder.get('DOI', ''),
            'award': '; '.join(funder.get('award', [])),
            'doi_asserted_by': funder.get('doi-asserted-by', '')
        }
        funders.append(funder_data)
    
    return funders


def extract_work_metadata(work: Dict, include_refs: bool = True) -> Dict:
    """
    Extract all available metadata from a CrossRef work record.
    """
    # Basic identifiers
    doi = work.get('DOI', '')
    
    # Title handling (can be list)
    titles = work.get('title', [])
    title = titles[0] if titles else ''
    subtitle = work.get('subtitle', [])
    subtitle = subtitle[0] if subtitle else ''
    
    # Original title (for translations)
    original_title = work.get('original-title', [])
    original_title = original_title[0] if original_title else ''
    
    # Short title
    short_title = work.get('short-title', [])
    short_title = short_title[0] if short_title else ''
    
    # Container (journal/book) info
    container_title = work.get('container-title', [])
    container_title = container_title[0] if container_title else ''
    short_container = work.get('short-container-title', [])
    short_container = short_container[0] if short_container else ''
    
    # ISSNs
    issns = work.get('ISSN', [])
    issn_types = work.get('issn-type', [])
    print_issn = ''
    electronic_issn = ''
    for issn_info in issn_types:
        if issn_info.get('type') == 'print':
            print_issn = issn_info.get('value', '')
        elif issn_info.get('type') == 'electronic':
            electronic_issn = issn_info.get('value', '')
    
    # ISBNs
    isbns = work.get('ISBN', [])
    
    # Dates
    pub_date, pub_year, pub_month, pub_day = parse_date_parts(work.get('published', {}).get('date-parts'))
    pub_print, print_year, _, _ = parse_date_parts(work.get('published-print', {}).get('date-parts'))
    pub_online, online_year, _, _ = parse_date_parts(work.get('published-online', {}).get('date-parts'))
    created, _, _, _ = parse_date_parts(work.get('created', {}).get('date-parts'))
    deposited, _, _, _ = parse_date_parts(work.get('deposited', {}).get('date-parts'))
    indexed, _, _, _ = parse_date_parts(work.get('indexed', {}).get('date-parts'))
    
    # Authors and contributors
    authors = parse_authors(work.get('author', []))
    editors = parse_authors(work.get('editor', []))
    translators = parse_authors(work.get('translator', []))
    
    # References
    references = parse_references(work.get('reference', [])) if include_refs else []
    
    # Funders
    funders = parse_funders(work.get('funder', []))
    
    # License info
    licenses = work.get('license', [])
    license_urls = '; '.join([lic.get('URL', '') for lic in licenses])
    
    # Links
    links = work.get('link', [])
    pdf_link = ''
    for link in links:
        if link.get('content-type') == 'application/pdf':
            pdf_link = link.get('URL', '')
            break
    
    # Subject categories
    subjects = work.get('subject', [])
    
    # Abstract
    abstract = work.get('abstract', '')
    # Clean JATS XML tags from abstract
    if abstract:
        abstract = re.sub(r'<[^>]+>', '', abstract).strip()
    
    # Build flattened author strings for CSV
    author_names = '; '.join([a['display_name'] for a in authors if a['display_name']])
    author_orcids = '; '.join([a['orcid'] for a in authors if a['orcid']])
    author_affiliations = ' | '.join([a['affiliation'] for a in authors if a['affiliation']])
    
    # Build output dict
    metadata = {
        # Identifiers
        'doi': doi,
        'doi_url': f"https://doi.org/{doi}" if doi else '',
        
        # Titles
        'title': title,
        'subtitle': subtitle,
        'original_title': original_title,
        'short_title': short_title,
        
        # Type
        'type': work.get('type', ''),
        'subtype': work.get('subtype', ''),
        
        # Container (journal/book)
        'container_title': container_title,
        'short_container_title': short_container,
        'issn': '; '.join(issns) if issns else '',
        'issn_print': print_issn,
        'issn_electronic': electronic_issn,
        'isbn': '; '.join(isbns) if isbns else '',
        
        # Volume/Issue/Pages
        'volume': work.get('volume', ''),
        'issue': work.get('issue', ''),
        'page': work.get('page', ''),
        'article_number': work.get('article-number', ''),
        
        # Dates
        'published_date': pub_date,
        'published_year': pub_year,
        'published_month': pub_month,
        'published_day': pub_day,
        'published_print': pub_print,
        'published_online': pub_online,
        'created_date': created,
        'deposited_date': deposited,
        'indexed_date': indexed,
        
        # Authors (flattened for CSV)
        'authors': author_names,
        'author_count': len(authors),
        'author_orcids': author_orcids,
        'author_affiliations': author_affiliations,
        
        # Editors
        'editors': '; '.join([e['display_name'] for e in editors if e['display_name']]),
        
        # Abstract
        'abstract': abstract,
        
        # Publisher
        'publisher': work.get('publisher', ''),
        'publisher_location': work.get('publisher-location', ''),
        
        # References
        'reference_count': work.get('reference-count', 0),
        'references_doi_count': len([r for r in references if r.get('doi')]),
        
        # Citations
        'is_referenced_by_count': work.get('is-referenced-by-count', 0),
        
        # Subjects
        'subjects': '; '.join(subjects),
        
        # Funding
        'funder_count': len(funders),
        'funders': '; '.join([f['name'] for f in funders if f['name']]),
        
        # License
        'license': license_urls,
        
        # Links
        'url': work.get('URL', ''),
        'pdf_url': pdf_link,
        'resource_url': work.get('resource', {}).get('primary', {}).get('URL', ''),
        
        # Language
        'language': work.get('language', ''),
        
        # Scores
        'score': work.get('score', ''),
        
        # Source
        'source': work.get('source', ''),
        'member': work.get('member', ''),
        'prefix': work.get('prefix', ''),
        
        # Full nested data for JSON export
        '_authors_full': authors,
        '_editors_full': editors,
        '_translators_full': translators,
        '_references_full': references if include_refs else [],
        '_funders_full': funders,
        '_raw': work  # keep original for debugging
    }
    
    return metadata


print("✅ Helper functions loaded.")

## CrossRef API Functions

In [ ]:
def query_crossref_by_orcid(orcid: str, progress_callback=None) -> Tuple[List[Dict], Dict]:
    """
    Query CrossRef for all works linked to an ORCID.
    
    Args:
        orcid: ORCID identifier (e.g., 0000-0002-1825-0097)
        progress_callback: Optional function to report progress
    
    Returns:
        (list of work metadata dicts, query stats dict)
    """
    orcid = clean_orcid(orcid)
    if not orcid:
        return [], {'error': 'Invalid ORCID format'}
    
    works = []
    stats = {
        'orcid': orcid,
        'total_results': 0,
        'pages_fetched': 0,
        'errors': [],
        'start_time': datetime.now().isoformat()
    }
    
    # Build query URL
    base_url = f"{config.CROSSREF_API_BASE}/works"
    
    # First request to get total count
    params = {
        'filter': f'orcid:{orcid}',
        'rows': config.ROWS_PER_PAGE,
        'offset': 0,
        'mailto': 'matt@mattartz.me'  # polite pool access
    }
    
    headers = {'User-Agent': config.USER_AGENT}
    
    try:
        response = requests.get(base_url, params=params, headers=headers, timeout=config.TIMEOUT)
        response.raise_for_status()
        data = response.json()
        
        total_results = data.get('message', {}).get('total-results', 0)
        stats['total_results'] = total_results
        
        if total_results == 0:
            print(f"No works found for ORCID {orcid}")
            return [], stats
        
        print(f"Found {total_results} works for ORCID {orcid}")
        
        # Process first page
        items = data.get('message', {}).get('items', [])
        for item in items:
            works.append(extract_work_metadata(item, config.INCLUDE_REFERENCES))
        stats['pages_fetched'] = 1
        
        if progress_callback:
            progress_callback(len(works), total_results)
        
        # Fetch remaining pages
        offset = config.ROWS_PER_PAGE
        page = 2
        
        while offset < total_results and page <= config.MAX_PAGES:
            time.sleep(config.REQUEST_DELAY)
            
            params['offset'] = offset
            
            try:
                response = requests.get(base_url, params=params, headers=headers, timeout=config.TIMEOUT)
                response.raise_for_status()
                data = response.json()
                
                items = data.get('message', {}).get('items', [])
                if not items:
                    break
                
                for item in items:
                    works.append(extract_work_metadata(item, config.INCLUDE_REFERENCES))
                
                stats['pages_fetched'] = page
                
                if progress_callback:
                    progress_callback(len(works), total_results)
                
                print(f"  Page {page}: {len(works)}/{total_results} works")
                
            except Exception as e:
                stats['errors'].append(f"Page {page}: {str(e)}")
                print(f"  Error on page {page}: {e}")
            
            offset += config.ROWS_PER_PAGE
            page += 1
        
    except Exception as e:
        stats['errors'].append(f"Initial query: {str(e)}")
        print(f"Error querying CrossRef: {e}")
    
    stats['end_time'] = datetime.now().isoformat()
    stats['works_retrieved'] = len(works)
    
    return works, stats


def query_crossref_by_doi(doi: str) -> Optional[Dict]:
    """
    Query CrossRef for a single work by DOI.
    
    Args:
        doi: DOI identifier (e.g., 10.1234/example)
    
    Returns:
        Work metadata dict or None if not found
    """
    doi = clean_doi(doi)
    if not doi:
        return None
    
    url = f"{config.CROSSREF_API_BASE}/works/{doi}"
    headers = {'User-Agent': config.USER_AGENT}
    params = {'mailto': 'matt@mattartz.me'}
    
    try:
        response = requests.get(url, params=params, headers=headers, timeout=config.TIMEOUT)
        
        if response.status_code == 404:
            return None
        
        response.raise_for_status()
        data = response.json()
        
        work = data.get('message', {})
        if work:
            return extract_work_metadata(work, config.INCLUDE_REFERENCES)
        return None
        
    except Exception as e:
        print(f"Error fetching DOI {doi}: {e}")
        return None


def query_crossref_by_doi_list(dois: List[str], progress_callback=None) -> Tuple[List[Dict], Dict]:
    """
    Query CrossRef for multiple works by DOI list.
    
    Args:
        dois: List of DOI identifiers
        progress_callback: Optional function to report progress
    
    Returns:
        (list of work metadata dicts, query stats dict)
    """
    works = []
    stats = {
        'total_dois': len(dois),
        'found': 0,
        'not_found': 0,
        'errors': [],
        'not_found_dois': [],
        'start_time': datetime.now().isoformat()
    }
    
    for idx, doi in enumerate(dois):
        clean = clean_doi(doi)
        if not clean:
            stats['errors'].append(f"Invalid DOI format: {doi}")
            continue
        
        work = query_crossref_by_doi(clean)
        
        if work:
            works.append(work)
            stats['found'] += 1
            print(f"[{idx+1}/{len(dois)}] ✓ {clean[:50]}")
        else:
            stats['not_found'] += 1
            stats['not_found_dois'].append(clean)
            print(f"[{idx+1}/{len(dois)}] ✗ {clean[:50]} (not found)")
        
        if progress_callback:
            progress_callback(idx + 1, len(dois))
        
        # Rate limiting
        if idx < len(dois) - 1:
            time.sleep(config.REQUEST_DELAY)
    
    stats['end_time'] = datetime.now().isoformat()
    stats['works_retrieved'] = len(works)
    
    return works, stats


print("✅ CrossRef API functions loaded.")

## Test API Connection

In [ ]:
# Test CrossRef API with a known DOI
print("Testing CrossRef API connection...")
print()

test_doi = "10.1111/j.1548-1433.2012.01515.x"  # Example anthropology article
print(f"Testing with DOI: {test_doi}")

test_work = query_crossref_by_doi(test_doi)

if test_work:
    print()
    print("✅ API connection successful!")
    print()
    print(f"Title: {test_work['title'][:80]}..." if len(test_work.get('title', '')) > 80 else f"Title: {test_work.get('title', '')}")
    print(f"Authors: {test_work['authors'][:60]}..." if len(test_work.get('authors', '')) > 60 else f"Authors: {test_work.get('authors', '')}")
    print(f"Journal: {test_work.get('container_title', '')}")
    print(f"Year: {test_work.get('published_year', '')}")
    print(f"Type: {test_work.get('type', '')}")
    print(f"Citations: {test_work.get('is_referenced_by_count', 0)}")
else:
    print("❌ API test failed - could not retrieve test DOI")

## Query Interface

*Choose your query method: ORCID search or DOI list lookup.*

In [ ]:
# Global results storage
query_results = []
query_stats = {}

# Create interface
query_mode = widgets.RadioButtons(
    options=['ORCID Search', 'DOI List'],
    value='ORCID Search',
    description='Query Mode:',
    style={'description_width': '100px'}
)

orcid_input = widgets.Text(
    placeholder='0000-0002-1825-0097',
    description='ORCID:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='400px')
)

doi_textarea = widgets.Textarea(
    placeholder='Enter DOIs (one per line):\n10.1234/example1\n10.5678/example2',
    description='DOI List:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='500px', height='150px')
)

doi_upload = widgets.FileUpload(
    accept='.csv,.txt',
    multiple=False,
    description='Or upload file'
)

run_button = widgets.Button(
    description='Run Query',
    button_style='primary',
    icon='search'
)

progress = widgets.IntProgress(
    value=0,
    min=0,
    max=100,
    description='Progress:',
    bar_style='info'
)

output = widgets.Output()

# Visibility toggle
orcid_box = widgets.VBox([orcid_input])
doi_box = widgets.VBox([doi_textarea, widgets.HTML('<b>Or upload CSV/TXT with DOIs:</b>'), doi_upload])
doi_box.layout.display = 'none'

def on_mode_change(change):
    if change['new'] == 'ORCID Search':
        orcid_box.layout.display = 'block'
        doi_box.layout.display = 'none'
    else:
        orcid_box.layout.display = 'none'
        doi_box.layout.display = 'block'

query_mode.observe(on_mode_change, names='value')

def update_progress(current, total):
    progress.max = total
    progress.value = current

def run_query(button):
    global query_results, query_stats
    query_results = []
    query_stats = {}
    
    with output:
        clear_output()
        progress.value = 0
        
        if query_mode.value == 'ORCID Search':
            orcid = orcid_input.value.strip()
            if not orcid:
                print("⚠️ Please enter an ORCID.")
                return
            
            print(f"🔍 Querying CrossRef for ORCID: {orcid}")
            print("=" * 60)
            print()
            
            query_results, query_stats = query_crossref_by_orcid(orcid, update_progress)
            
        else:  # DOI List
            dois = []
            
            # Check for uploaded file first
            if doi_upload.value:
                file_info = list(doi_upload.value.values())[0]
                content = file_info['content'].decode('utf-8')
                
                if file_info['metadata']['name'].endswith('.csv'):
                    # Try to find DOI column
                    df = pd.read_csv(StringIO(content))
                    doi_col = None
                    for col in df.columns:
                        if 'doi' in col.lower():
                            doi_col = col
                            break
                    if doi_col:
                        dois = df[doi_col].dropna().tolist()
                    else:
                        # Use first column
                        dois = df.iloc[:, 0].dropna().tolist()
                else:
                    # Plain text - one DOI per line
                    dois = [line.strip() for line in content.split('\n') if line.strip()]
            else:
                # Use textarea input
                dois = [line.strip() for line in doi_textarea.value.split('\n') if line.strip()]
            
            if not dois:
                print("⚠️ Please enter DOIs or upload a file.")
                return
            
            print(f"🔍 Querying CrossRef for {len(dois)} DOIs")
            print("=" * 60)
            print()
            
            query_results, query_stats = query_crossref_by_doi_list(dois, update_progress)
        
        # Display summary
        print()
        print("=" * 60)
        print("📊 QUERY SUMMARY")
        print("=" * 60)
        print(f"Works retrieved: {len(query_results)}")
        
        if query_results:
            types = {}
            for work in query_results:
                t = work.get('type', 'unknown')
                types[t] = types.get(t, 0) + 1
            
            print(f"\nBy type:")
            for t, count in sorted(types.items(), key=lambda x: -x[1]):
                print(f"  • {t}: {count}")
            
            years = [w.get('published_year') for w in query_results if w.get('published_year')]
            if years:
                print(f"\nYear range: {min(years)} - {max(years)}")
            
            total_citations = sum(w.get('is_referenced_by_count', 0) for w in query_results)
            print(f"Total citations: {total_citations:,}")
        
        if query_stats.get('errors'):
            print(f"\n⚠️ Errors: {len(query_stats['errors'])}")
        
        print()
        print("✅ Query complete! Run the export cell to download results.")

run_button.on_click(run_query)

# Display interface
display(widgets.HTML('<h3>🔎 CrossRef Query</h3>'))
display(query_mode)
display(orcid_box)
display(doi_box)
display(widgets.HBox([run_button]))
display(progress)
display(output)

## Preview Results

In [ ]:
# Preview query results
if query_results:
    print(f"📋 Preview of {len(query_results)} works")
    print("=" * 80)
    print()
    
    # Create preview dataframe (exclude internal fields)
    preview_cols = ['doi', 'title', 'authors', 'container_title', 'published_year', 'type', 'is_referenced_by_count']
    preview_data = [{k: v for k, v in w.items() if k in preview_cols} for w in query_results]
    preview_df = pd.DataFrame(preview_data)
    
    # Truncate long strings for display
    preview_df['title'] = preview_df['title'].str[:60] + '...'
    preview_df['authors'] = preview_df['authors'].str[:40] + '...'
    
    display(preview_df.head(20))
    
    if len(query_results) > 20:
        print(f"\n... and {len(query_results) - 20} more works")
else:
    print("⚠️ No results to preview. Run the query cell first.")

## Export Results

In [ ]:
def export_results(works: List[Dict], stats: Dict) -> Tuple[str, str, str]:
    """
    Export query results to CSV and JSON files.
    
    Returns:
        (csv_path, json_path, stats_path)
    """
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # Determine filename base
    if stats.get('orcid'):
        orcid_slug = stats['orcid'].replace('-', '')
        base_name = f"crossref_orcid_{orcid_slug}"
    else:
        base_name = f"crossref_doi_batch"
    
    # --- CSV Export (flattened) ---
    csv_cols = [
        'doi', 'doi_url', 'title', 'subtitle', 'original_title',
        'type', 'subtype',
        'container_title', 'short_container_title',
        'issn', 'issn_print', 'issn_electronic', 'isbn',
        'volume', 'issue', 'page', 'article_number',
        'published_date', 'published_year', 'published_month', 'published_day',
        'published_print', 'published_online',
        'authors', 'author_count', 'author_orcids', 'author_affiliations',
        'editors',
        'abstract',
        'publisher', 'publisher_location',
        'reference_count', 'references_doi_count',
        'is_referenced_by_count',
        'subjects',
        'funder_count', 'funders',
        'license',
        'url', 'pdf_url', 'resource_url',
        'language',
        'source', 'member', 'prefix'
    ]
    
    csv_data = [{k: w.get(k, '') for k in csv_cols} for w in works]
    df = pd.DataFrame(csv_data)
    
    csv_filename = f"{base_name}_{timestamp}.csv"
    csv_path = os.path.join(config.OUTPUT_PATH, csv_filename)
    df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    
    print(f"✅ CSV exported: {csv_filename}")
    print(f"   Rows: {len(df)}")
    print(f"   Columns: {len(csv_cols)}")
    
    # --- JSON Export (full nested structure) ---
    json_data = {
        'metadata': {
            'query_stats': stats,
            'export_timestamp': timestamp,
            'work_count': len(works)
        },
        'works': []
    }
    
    for work in works:
        # Create clean export (without _raw to save space)
        work_export = {k: v for k, v in work.items() if k != '_raw'}
        json_data['works'].append(work_export)
    
    json_filename = f"{base_name}_{timestamp}.json"
    json_path = os.path.join(config.OUTPUT_PATH, json_filename)
    
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(json_data, f, indent=2, ensure_ascii=False, default=str)
    
    print(f"\n✅ JSON exported: {json_filename}")
    print(f"   Includes full author details, references, funders")
    
    # --- Stats Export ---
    stats_filename = f"{base_name}_{timestamp}_stats.json"
    stats_path = os.path.join(config.OUTPUT_PATH, stats_filename)
    
    with open(stats_path, 'w', encoding='utf-8') as f:
        json.dump(stats, f, indent=2, default=str)
    
    print(f"\n✅ Stats exported: {stats_filename}")
    
    return csv_path, json_path, stats_path


# Execute export
if query_results:
    print("💾 Exporting Query Results")
    print("=" * 60)
    print()
    
    csv_path, json_path, stats_path = export_results(query_results, query_stats)
    
    print()
    print("=" * 60)
    print("🎉 Export complete!")
else:
    print("⚠️ No results to export. Run the query cell first.")

## Download Files

In [ ]:
# Download exported files
def create_download_interface():
    """Create interface for downloading exported files."""
    
    if not os.path.exists(config.OUTPUT_PATH):
        print("⚠️ No output directory found. Run the export cell first.")
        return
    
    files_list = os.listdir(config.OUTPUT_PATH)
    if not files_list:
        print("⚠️ No exported files found. Run the export cell first.")
        return
    
    print("📥 Available Files for Download")
    print("=" * 60)
    print()
    
    # Group by type
    csv_files = sorted([f for f in files_list if f.endswith('.csv')])
    json_files = sorted([f for f in files_list if f.endswith('.json') and not f.endswith('_stats.json')])
    stats_files = sorted([f for f in files_list if f.endswith('_stats.json')])
    
    all_files = csv_files + json_files + stats_files
    
    for f in all_files:
        file_path = os.path.join(config.OUTPUT_PATH, f)
        size_kb = os.path.getsize(file_path) / 1024
        
        if size_kb > 1024:
            size_str = f"{size_kb/1024:.1f} MB"
        else:
            size_str = f"{size_kb:.1f} KB"
        
        icon = "📊" if f.endswith('.csv') else "📋" if f.endswith('_stats.json') else "🗂️"
        print(f"{icon} {f} ({size_str})")
    
    print()
    
    if IN_COLAB:
        print("Click buttons to download:")
        print()
        
        for f in all_files:
            file_path = os.path.join(config.OUTPUT_PATH, f)
            
            button = widgets.Button(
                description=f'📥 {f[:45]}...' if len(f) > 45 else f'📥 {f}',
                tooltip=f'Download {f}',
                layout=widgets.Layout(width='450px', height='35px', margin='3px'),
                style={'button_color': '#6096BA'}
            )
            
            def make_handler(path):
                def handler(b):
                    files.download(path)
                return handler
            
            button.on_click(make_handler(file_path))
            display(button)
    else:
        print(f"Files saved to: {config.OUTPUT_PATH}")

create_download_interface()

## Analysis & Visualization (Optional)

In [ ]:
# Additional analysis of query results
if query_results:
    print("📈 Publication Analysis")
    print("=" * 60)
    print()
    
    # Publications by year
    years = [w.get('published_year') for w in query_results if w.get('published_year')]
    if years:
        year_counts = pd.Series(years).value_counts().sort_index()
        
        print("📅 Publications by Year")
        print("-" * 40)
        for year, count in year_counts.items():
            bar = "█" * min(count, 50)
            print(f"{year}: {bar} {count}")
        print()
    
    # Top journals
    journals = [w.get('container_title') for w in query_results if w.get('container_title')]
    if journals:
        journal_counts = pd.Series(journals).value_counts().head(10)
        
        print("📚 Top 10 Journals/Venues")
        print("-" * 40)
        for journal, count in journal_counts.items():
            print(f"  {count:3d} | {journal[:50]}")
        print()
    
    # Co-authors (from author lists)
    all_authors = []
    for w in query_results:
        for author in w.get('_authors_full', []):
            if author.get('display_name'):
                all_authors.append(author['display_name'])
    
    if all_authors:
        author_counts = pd.Series(all_authors).value_counts().head(15)
        
        print("👥 Top 15 Authors (by frequency)")
        print("-" * 40)
        for author, count in author_counts.items():
            print(f"  {count:3d} | {author[:40]}")
        print()
    
    # Citation statistics
    citations = [w.get('is_referenced_by_count', 0) for w in query_results]
    if citations:
        print("📊 Citation Statistics")
        print("-" * 40)
        print(f"  Total citations: {sum(citations):,}")
        print(f"  Mean per work: {sum(citations)/len(citations):.1f}")
        print(f"  Median: {sorted(citations)[len(citations)//2]}")
        print(f"  Max: {max(citations)}")
        
        # h-index calculation
        sorted_cites = sorted(citations, reverse=True)
        h_index = 0
        for i, c in enumerate(sorted_cites, 1):
            if c >= i:
                h_index = i
            else:
                break
        print(f"  h-index: {h_index}")
        print()
    
    # Works with ORCIDs
    orcid_authors = 0
    for w in query_results:
        for author in w.get('_authors_full', []):
            if author.get('orcid'):
                orcid_authors += 1
    
    print("🆔 ORCID Coverage")
    print("-" * 40)
    print(f"  Authors with ORCID: {orcid_authors}")
    
else:
    print("⚠️ No results to analyze. Run the query cell first.")

## Wikidata Preparation (Optional)

*Export results in a format suitable for the AAA Wikidata Article Importer.*

In [ ]:
def export_for_wikidata(works: List[Dict]) -> str:
    """
    Export works in format compatible with AAA_Wikidata_Article_Importer.
    """
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # Map to importer expected columns
    wikidata_rows = []
    for w in works:
        # Build author string in expected format (semicolon separated)
        authors = '; '.join([a['display_name'] for a in w.get('_authors_full', []) if a.get('display_name')])
        
        row = {
            'DOI': w.get('doi', ''),
            'Title': w.get('title', ''),
            'Authors': authors,
            'Publication Date': w.get('published_date', ''),
            'Year': w.get('published_year', ''),
            'Journal': w.get('container_title', ''),
            'ISSN': w.get('issn', ''),
            'Volume': w.get('volume', ''),
            'Issue': w.get('issue', ''),
            'Page': w.get('page', ''),
            'URL': w.get('url', ''),
            'Type': w.get('type', '')
        }
        wikidata_rows.append(row)
    
    df = pd.DataFrame(wikidata_rows)
    
    filename = f"crossref_for_wikidata_{timestamp}.csv"
    filepath = os.path.join(config.OUTPUT_PATH, filename)
    df.to_csv(filepath, index=False, encoding='utf-8-sig')
    
    print(f"✅ Wikidata-ready CSV exported: {filename}")
    print(f"   Works: {len(df)}")
    print(f"   Compatible with AAA_Wikidata_Article_Importer")
    
    return filepath


if query_results:
    print("🔗 Exporting for Wikidata Import")
    print("=" * 60)
    print()
    
    wikidata_path = export_for_wikidata(query_results)
    
    if IN_COLAB:
        print()
        download_btn = widgets.Button(
            description='📥 Download Wikidata CSV',
            button_style='success',
            layout=widgets.Layout(width='300px')
        )
        
        def download_wikidata(b):
            files.download(wikidata_path)
        
        download_btn.on_click(download_wikidata)
        display(download_btn)
else:
    print("⚠️ No results to export. Run the query cell first.")